# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barak-hub/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:

from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd

hf_token = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset",
    token=hf_token
)

df = pd.read_parquet(file_path)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))


fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Dataset loaded successfully!
Rows: 2414248
Columns: 21


In [2]:

# Select important search-performance metrics
metrics = [
    "impressions_90d",
    "clicks_90d",
    "impressions_prev30",
    "clicks_prev30"
]

# Calculate distribution statistics
distribution = df[metrics].describe(
    percentiles=[0.50, 0.90, 0.99]
)

print(distribution.round(2).to_string())


       impressions_90d  clicks_90d  impressions_prev30  clicks_prev30
count       2414248.00  2414248.00          2414248.00     2414248.00
mean             87.71        0.19               31.16           0.07
std             933.27        1.94              424.27           0.77
min              10.00        0.00                0.00           0.00
50%              23.00        0.00                7.00           0.00
90%             141.00        0.00               50.00           0.00
99%             964.53        3.00              359.00           2.00
max          543044.00     1109.00           280503.00         357.00


In [3]:

# Find the 99th percentile threshold
threshold = df["impressions_90d"].quantile(0.99)

# Select records above that threshold
top_1_percent = df[
    df["impressions_90d"] > threshold
]

# Calculate their share of total impressions
share = (
    top_1_percent["impressions_90d"].sum()
    / df["impressions_90d"].sum()
) * 100

print("99th percentile threshold:", round(threshold, 2))
print("Records above threshold:", len(top_1_percent))
print("Share of total impressions:", round(share, 2), "%")


99th percentile threshold: 964.53
Records above threshold: 24143
Share of total impressions: 35.13 %


### Distribution Audit Findings

The FlyRank dataset contains 2,414,248 records. Search impressions and clicks exhibit strongly right-skewed distributions.

The median number of 90-day impressions is 23, compared with a mean of 87.71 and a maximum of 543,044.

The 99th percentile for impressions is 964.53. The 24,143 records above this threshold account for 35.13% of total impressions, indicating substantial concentration among high-volume records.

The median number of clicks is zero, indicating that at least half of the records received no clicks.

**Conclusion:** Search-performance distributions are highly skewed. Model evaluation should consider both high-volume and typical low-volume records to avoid misleading conclusions.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [5]:

# Keep records with sufficient historical impressions
audit = df[df["impressions_prev30"] >= 30].copy()

# Calculate historical CTR
audit["ctr_prev30"] = (
    audit["clicks_prev30"] /
    audit["impressions_prev30"]
)

# Group records by historical ranking
audit["rank_group"] = pd.cut(
    audit["avg_position_prev30"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["Top 3", "4-10", "11-20", "21+"]
)

# Calculate CTR for each ranking group
results = audit.groupby(
    "rank_group", observed=True
).agg(
    impressions=("impressions_prev30", "sum"),
    clicks=("clicks_prev30", "sum"),
    records=("ctr_prev30", "size")
)

results["CTR (%)"] = (
    results["clicks"] / results["impressions"] * 100
)

print(results.round(2).to_string())


            impressions  clicks  records  CTR (%)
rank_group                                       
Top 3           8762732   47509    43809     0.54
4-10           38673296   72835   224677     0.19
11-20           6906112    7157    56567     0.10
21+             5928627    2516    64393     0.04


### Signal Test #1 — Historical Search Ranking vs. CTR

**Hypothesis:** Queries with better historical search rankings tend to have higher historical click-through rates.

**Method:** Filtered records with at least 30 impressions in the previous 30 days, grouped them by historical average search position, and calculated impression-weighted CTR for each group.

**Results:**
- Top 3: 0.54% CTR
- Positions 4–10: 0.19% CTR
- Positions 11–20: 0.10% CTR
- Positions 21+: 0.04% CTR

**Verdict: CONFIRMED (descriptive association).**

CTR decreases consistently as numerical search position increases. However, the result does not establish a causal relationship, and 907 eligible records were not assigned to the ranking groups.

**Business implication:** Historical search position is a useful signal to consider when identifying potential engagement opportunities, but ranking alone should not determine recommendations.

In [6]:

# Group records by historical impression volume
audit["impression_group"] = pd.cut(
    audit["impressions_prev30"],
    bins=[29, 50, 100, 500, float("inf")],
    labels=["30-50", "51-100", "101-500", "501+"]
)

# Calculate engagement for each group
results2 = audit.groupby(
    "impression_group", observed=True
).agg(
    impressions=("impressions_prev30", "sum"),
    clicks=("clicks_prev30", "sum"),
    records=("impressions_prev30", "size")
)

results2["CTR (%)"] = (
    results2["clicks"] / results2["impressions"] * 100
)

print(results2.round(3).to_string())


                  impressions  clicks  records  CTR (%)
impression_group                                       
30-50                 5791189   14033   150980    0.242
51-100                8573785   21522   121432    0.251
101-500              20038074   52928   102462    0.264
501+                 25970492   41639    15479    0.160


### Signal Test #2 — Historical Impression Volume vs. CTR

**Hypothesis:** Search queries with more historical impressions tend to have higher click-through rates.

**Method:** Filtered records with at least 30 historical impressions, divided them into four impression-volume groups, and calculated impression-weighted CTR for each group.

**Results:**
- 30–50 impressions: 0.242% CTR
- 51–100 impressions: 0.251% CTR
- 101–500 impressions: 0.264% CTR
- 501+ impressions: 0.160% CTR

**Verdict: MIXED.**

CTR increases slightly across the first three impression-volume groups but decreases substantially in the highest-volume group.

**Business implication:** High impression volume alone is not a reliable indicator of strong engagement. Search ranking, query relevance, and other contextual factors should also be considered when prioritizing opportunities.

**Limitation:** These are descriptive, aggregated results. They do not establish a causal relationship between impression volume and CTR.

In [7]:

# Group queries by number of tokens
audit["query_length_group"] = pd.cut(
    audit["query_token_count"],
    bins=[-1, 2, 4, 7, float("inf")],
    labels=["0-2 tokens", "3-4 tokens",
            "5-7 tokens", "8+ tokens"]
)

# Calculate historical engagement
results3 = audit.groupby(
    "query_length_group", observed=True
).agg(
    impressions=("impressions_prev30", "sum"),
    clicks=("clicks_prev30", "sum"),
    records=("query_token_count", "size")
)

results3["CTR (%)"] = (
    results3["clicks"] /
    results3["impressions"] * 100
)

print(results3.round(3).to_string())


                    impressions  clicks  records  CTR (%)
query_length_group                                       
0-2 tokens              4541212   11610    22825    0.256
3-4 tokens             20934938   44735   131523    0.214
5-7 tokens             26549497   56379   176870    0.212
8+ tokens               8347893   17398    59135    0.208


### Signal Test #3 — Query Length vs. Historical CTR

**Hypothesis:** Longer search queries tend to have higher historical click-through rates.

**Method:** Filtered records with at least 30 historical impressions, grouped queries by token count, and calculated impression-weighted CTR for each group.

**Results:**
- 0–2 tokens: 0.256% CTR
- 3–4 tokens: 0.214% CTR
- 5–7 tokens: 0.212% CTR
- 8+ tokens: 0.208% CTR

**Verdict: OPPOSITE.**

Historical CTR decreases as query length increases across the four groups. The shortest queries have the highest observed CTR, contrary to the original hypothesis.

**Business implication:** Longer queries should not automatically be prioritized as higher-engagement opportunities. Query length should be evaluated alongside historical ranking, impressions, and other relevant features.

**Limitation:** This is a descriptive association based on aggregated historical data. It does not establish causation or control for differences between clients and search positions.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [8]:

# Explore a proxy for CTR improvement opportunities
flag_audit = audit.copy()

# Calculate future-period CTR
flag_audit["ctr_last30"] = (
    flag_audit["clicks_last30"] /
    flag_audit["impressions_last30"].replace(0, float("nan"))
)

# Define a low historical CTR group
flag_audit["low_ctr_proxy"] = (
    flag_audit["ctr_prev30"] < 0.002
)

# Compare historical and future CTR
summary = flag_audit.groupby("low_ctr_proxy").agg(
    records=("ctr_prev30", "size"),
    historical_ctr=("ctr_prev30", "mean"),
    future_ctr=("ctr_last30", "mean")
)

print(summary.round(5).to_string())


               records  historical_ctr  future_ctr
low_ctr_proxy                                     
False            58809         0.01652     0.00672
True            331544         0.00001     0.00186


In [9]:

# Check records with and without future impressions
coverage = flag_audit.groupby("low_ctr_proxy").agg(
    total_records=("impressions_last30", "size"),
    zero_future_impressions=(
        "impressions_last30",
        lambda x: (x == 0).sum()
    ),
    positive_future_impressions=(
        "impressions_last30",
        lambda x: (x > 0).sum()
    )
)

coverage["zero_impression_pct"] = (
    coverage["zero_future_impressions"]
    / coverage["total_records"] * 100
)

print(coverage.round(2).to_string())


               total_records  zero_future_impressions  positive_future_impressions  zero_impression_pct
low_ctr_proxy                                                                                          
False                  58809                     1449                        57360                 2.46
True                  331544                    26341                       305203                 7.94


### Flag-Linked Test — `needs_ctr_fix`

**Flag investigated:** FlyRank's `needs_ctr_fix`, which relates to identifying CTR improvement opportunities.

**Hypothesis:** Records with low historical CTR may represent opportunities for future engagement improvement.

**Method:** Used records with at least 30 historical impressions. Defined an exploratory low-CTR proxy using historical CTR below 0.2%. This threshold was selected for analysis and is not FlyRank's official flag definition.

**Results:**
- Low-CTR proxy group: 331,544 records; historical mean CTR 0.001%; future mean CTR 0.186%.
- Higher-CTR group: 58,809 records; historical mean CTR 1.652%; future mean CTR 0.672%.
- Zero future impressions occurred in 7.94% of low-CTR records and 2.46% of higher-CTR records.

**Verdict: MIXED.**

Low-historical-CTR records showed higher average CTR in the later period, suggesting potential for improvement. However, this does not demonstrate that the actual FlyRank flag is effective or that any recommended intervention would cause improvement.

**Limitations:** The product's actual flag values and thresholds are unavailable. Future CTR is undefined for records with zero future impressions, and the averages are unweighted. Regression to the mean and differences in query characteristics may also influence the results.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### Practical Recommendations for FlyRank's Content Team

FlyRank's content team should prioritize search engagement opportunities by considering historical ranking, impression volume, and CTR together rather than relying on a single metric. Pages or queries with substantial impressions and weak CTR may deserve investigation, but low CTR alone does not prove that a content change will improve performance. Before recommending action, the team should review historical trends, account for differences between clients, and validate opportunities using later-period outcomes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.